# K-Nearest Neighbors (KNN) Algorithm

In this notebook, we'll explore the K-nearest neighbors algorithm, one of the most fundamental and intuitive machine learning algorithms. We'll cover both classification and regression tasks, implement KNN from scratch, and use scikit-learn's implementation.

## What is KNN?

K-nearest neighbors is a simple, instance-based, non-parametric algorithm that:
- Stores all training examples in memory
- Classifies new instances based on the majority vote of its 'k' closest neighbors
- For regression, predicts the average value of its 'k' closest neighbors

The algorithm makes no assumptions about the underlying data distribution, making it versatile for various applications.

## 1. Import Required Libraries

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_boston, load_breast_cancer, make_classification, make_regression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score
from matplotlib.colors import ListedColormap
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Set up plot style
plt.style.use('seaborn-whitegrid')
sns.set_palette("husl")

# For reproducibility
np.random.seed(42)

## 2. Understanding KNN Algorithm

The K-Nearest Neighbors algorithm is a supervised machine learning algorithm that can be used for both classification and regression tasks. It's based on the principle that similar data points exist close to each other in feature space.

### How KNN Works:

1. **Choose a value for K**: The number of neighbors to consider
2. **Find K-Nearest Neighbors**: Calculate the distance (usually Euclidean) between the new point and all training examples
3. **Vote or Average**: 
   - For classification: Take majority vote of K-Nearest Neighbors
   - For regression: Take average value of K-Nearest Neighbors
4. **Assign label or value**: Assign the majority class or average value to the new point

### Common Distance Metrics:

- **Euclidean Distance**: $\sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$
- **Manhattan Distance**: $\sum_{i=1}^{n}|x_i - y_i|$
- **Minkowski Distance**: $(\sum_{i=1}^{n}|x_i - y_i|^p)^{1/p}$
- **Hamming Distance**: Count of positions where corresponding elements are different

### The 'k' Parameter:

- **Small k**: Low bias, high variance (more sensitive to noise)
- **Large k**: High bias, low variance (smoother decision boundaries)
- **k = 1**: Memorizes the training data (potentially overfits)
- **k = n**: Predicts the majority class (potentially underfits)
- **Choosing k**: Often an odd number is selected to avoid ties in binary classification

## 3. Dataset Preparation

Let's prepare a dataset for demonstrating KNN. We'll use the Iris dataset for classification and generate a synthetic dataset for regression.

In [ ]:
# Load Iris dataset for classification
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Create a dataframe for easier exploration
iris_df = pd.DataFrame(X_iris, columns=feature_names)
iris_df['species'] = pd.Categorical.from_codes(y_iris, target_names)

# Display first few rows of the dataframe
print("Iris Dataset (first 5 rows):")
print(iris_df.head())

# Basic statistics
print("\nDataset Statistics:")
print(iris_df.describe())

# Check for missing values
print("\nMissing Values:", iris_df.isnull().sum().sum())

# Class distribution
print("\nClass Distribution:")
print(iris_df['species'].value_counts())

# Creating a synthetic dataset for regression
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)
reg_df = pd.DataFrame({'X': X_reg.flatten(), 'y': y_reg})

print("\nRegression Dataset (first 5 rows):")
print(reg_df.head())

In [ ]:
# Visualize the Iris dataset
plt.figure(figsize=(15, 6))

# Pairplot for Iris dataset
plt.subplot(1, 2, 1)
sns.scatterplot(x='sepal length (cm)', y='sepal width (cm)', 
                hue='species', style='species', data=iris_df)
plt.title('Iris Dataset - Sepal Features')

plt.subplot(1, 2, 2)
sns.scatterplot(x='petal length (cm)', y='petal width (cm)', 
                hue='species', style='species', data=iris_df)
plt.title('Iris Dataset - Petal Features')

plt.tight_layout()
plt.show()

# Visualize the synthetic regression dataset
plt.figure(figsize=(10, 6))
plt.scatter(reg_df['X'], reg_df['y'], alpha=0.7)
plt.title('Synthetic Regression Dataset')
plt.xlabel('X')
plt.ylabel('y')
plt.show()

### Data Preprocessing

Before applying KNN, we need to preprocess the data:
1. Split the data into training and testing sets
2. Scale the features (important for distance-based algorithms like KNN)

Let's perform these steps for both datasets:

In [ ]:
# For Iris Classification Dataset
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# Scale the features
scaler_iris = StandardScaler()
X_train_iris_scaled = scaler_iris.fit_transform(X_train_iris)
X_test_iris_scaled = scaler_iris.transform(X_test_iris)

print("Iris Classification Data:")
print(f"Training set shape: {X_train_iris_scaled.shape}")
print(f"Test set shape: {X_test_iris_scaled.shape}")

# For Regression Dataset
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

# Scale the features
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

print("\nRegression Data:")
print(f"Training set shape: {X_train_reg_scaled.shape}")
print(f"Test set shape: {X_test_reg_scaled.shape}")

## 4. Implementing KNN from Scratch

Let's implement the KNN algorithm from scratch to understand its inner workings. Our implementation will include:

1. Distance calculation functions (Euclidean, Manhattan)
2. Nearest neighbor finding
3. Classification and regression prediction methods

In [ ]:
class KNNFromScratch:
    def __init__(self, k=5, distance_metric='euclidean'):
        """
        Initialize KNN model
        
        Parameters:
        -----------
        k: int, default=5
            Number of neighbors to use
        distance_metric: str, default='euclidean'
            Distance metric to use. Options: 'euclidean', 'manhattan'
        """
        self.k = k
        self.distance_metric = distance_metric
        self.X_train = None
        self.y_train = None
        
    def fit(self, X, y):
        """Store training data"""
        self.X_train = X
        self.y_train = y
        return self
    
    def euclidean_distance(self, x1, x2):
        """Calculate Euclidean distance between two points"""
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def manhattan_distance(self, x1, x2):
        """Calculate Manhattan distance between two points"""
        return np.sum(np.abs(x1 - x2))
    
    def calculate_distance(self, x1, x2):
        """Calculate distance based on chosen metric"""
        if self.distance_metric == 'euclidean':
            return self.euclidean_distance(x1, x2)
        elif self.distance_metric == 'manhattan':
            return self.manhattan_distance(x1, x2)
        else:
            raise ValueError("Unsupported distance metric")
    
    def get_neighbors(self, x):
        """Find k nearest neighbors"""
        # Calculate distances
        distances = []
        for i in range(len(self.X_train)):
            distance = self.calculate_distance(x, self.X_train[i])
            distances.append((distance, i))
        
        # Sort distances and get indices of k nearest neighbors
        distances.sort(key=lambda x: x[0])
        neighbor_indices = [distances[i][1] for i in range(min(self.k, len(distances)))]
        
        return neighbor_indices
    
    def predict_classification(self, X):
        """Make classification predictions for X"""
        predictions = []
        
        for x in X:
            # Find nearest neighbors
            neighbor_indices = self.get_neighbors(x)
            
            # Get classes of neighbors
            neighbor_labels = [self.y_train[i] for i in neighbor_indices]
            
            # Majority vote
            unique_labels, counts = np.unique(neighbor_labels, return_counts=True)
            predictions.append(unique_labels[np.argmax(counts)])
            
        return np.array(predictions)
    
    def predict_regression(self, X):
        """Make regression predictions for X"""
        predictions = []
        
        for x in X:
            # Find nearest neighbors
            neighbor_indices = self.get_neighbors(x)
            
            # Get values of neighbors
            neighbor_values = [self.y_train[i] for i in neighbor_indices]
            
            # Take average
            predictions.append(np.mean(neighbor_values))
            
        return np.array(predictions)

In [ ]:
# Test our implementation on a subset of the Iris dataset
# Use a smaller subset for faster computation
X_subset = X_train_iris_scaled[:50]
y_subset = y_train_iris[:50]
X_test_subset = X_test_iris_scaled[:20]
y_test_subset = y_test_iris[:20]

# Initialize and fit our custom KNN model
custom_knn = KNNFromScratch(k=3)
custom_knn.fit(X_subset, y_subset)

# Make predictions
custom_predictions = custom_knn.predict_classification(X_test_subset)

# Evaluate the model
custom_accuracy = accuracy_score(y_test_subset, custom_predictions)
print(f"Custom KNN Accuracy: {custom_accuracy:.4f}")

# Compare with sklearn implementation
sklearn_knn = KNeighborsClassifier(n_neighbors=3)
sklearn_knn.fit(X_subset, y_subset)
sklearn_predictions = sklearn_knn.predict(X_test_subset)
sklearn_accuracy = accuracy_score(y_test_subset, sklearn_predictions)
print(f"Sklearn KNN Accuracy: {sklearn_accuracy:.4f}")

In [ ]:
# Test our implementation for regression
# Use a smaller subset for faster computation
X_reg_subset = X_train_reg_scaled[:50]
y_reg_subset = y_train_reg[:50]
X_test_reg_subset = X_test_reg_scaled[:20]
y_test_reg_subset = y_test_reg[:20]

# Initialize and fit our custom KNN model for regression
custom_knn_reg = KNNFromScratch(k=3)
custom_knn_reg.fit(X_reg_subset, y_reg_subset)

# Make predictions
custom_reg_predictions = custom_knn_reg.predict_regression(X_test_reg_subset)

# Evaluate the model
custom_mse = mean_squared_error(y_test_reg_subset, custom_reg_predictions)
print(f"Custom KNN Regression MSE: {custom_mse:.4f}")

# Compare with sklearn implementation
sklearn_knn_reg = KNeighborsRegressor(n_neighbors=3)
sklearn_knn_reg.fit(X_reg_subset, y_reg_subset)
sklearn_reg_predictions = sklearn_knn_reg.predict(X_test_reg_subset)
sklearn_mse = mean_squared_error(y_test_reg_subset, sklearn_reg_predictions)
print(f"Sklearn KNN Regression MSE: {sklearn_mse:.4f}")

plt.figure(figsize=(10, 6))
plt.scatter(X_reg_subset.flatten(), y_reg_subset, color='blue', label='Training data')
plt.scatter(X_test_reg_subset.flatten(), y_test_reg_subset, color='green', label='Test data')
plt.scatter(X_test_reg_subset.flatten(), custom_reg_predictions, color='red', marker='x', label='Custom KNN predictions')
plt.scatter(X_test_reg_subset.flatten(), sklearn_reg_predictions, color='purple', marker='+', label='Sklearn KNN predictions')
plt.legend()
plt.title('KNN Regression Comparison')
plt.xlabel('X (scaled)')
plt.ylabel('y')
plt.show()

## 5. Using Scikit-learn's KNN Implementation

Now let's use scikit-learn's implementation of KNN for both classification and regression tasks.

In [ ]:
# KNN Classification on the entire Iris dataset
knn_classifier = KNeighborsClassifier(n_neighbors=5)
knn_classifier.fit(X_train_iris_scaled, y_train_iris)

# Make predictions
y_pred_iris = knn_classifier.predict(X_test_iris_scaled)

# Evaluate the model
iris_accuracy = accuracy_score(y_test_iris, y_pred_iris)
print(f"KNN Classification Accuracy: {iris_accuracy:.4f}")

# Display confusion matrix
conf_matrix = confusion_matrix(y_test_iris, y_pred_iris)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test_iris, y_pred_iris, target_names=target_names))

# Predict probabilities for each class
y_proba = knn_classifier.predict_proba(X_test_iris_scaled)

# Print sample of probability outputs
print("\nSample Probability Predictions (first 5 examples):")
for i in range(5):
    print(f"Sample {i+1}: {y_proba[i]} -> Predicted class: {target_names[y_pred_iris[i]]}")

In [ ]:
# KNN Regression on the synthetic dataset
knn_regressor = KNeighborsRegressor(n_neighbors=5)
knn_regressor.fit(X_train_reg_scaled, y_train_reg)

# Make predictions
y_pred_reg = knn_regressor.predict(X_test_reg_scaled)

# Evaluate the model
reg_mse = mean_squared_error(y_test_reg, y_pred_reg)
reg_r2 = r2_score(y_test_reg, y_pred_reg)
print(f"KNN Regression Mean Squared Error: {reg_mse:.4f}")
print(f"KNN Regression R^2 Score: {reg_r2:.4f}")

# Visualize the results
plt.figure(figsize=(10, 6))

# Sort points for a smoother curve
X_test_sorted = X_test_reg.copy()
indices = np.argsort(X_test_sorted.flatten())
X_test_sorted = X_test_sorted[indices]
y_test_sorted = y_test_reg[indices]

# Get predictions for the sorted test points
X_test_sorted_scaled = scaler_reg.transform(X_test_sorted)
y_pred_sorted = knn_regressor.predict(X_test_sorted_scaled)

plt.scatter(X_train_reg, y_train_reg, color='blue', alpha=0.5, label='Training data')
plt.scatter(X_test_reg, y_test_reg, color='green', alpha=0.5, label='Test data')
plt.plot(X_test_sorted, y_pred_sorted, color='red', linewidth=2, label='KNN Prediction')
plt.title('KNN Regression (k=5)')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()

## 6. Parameter Tuning - Finding Optimal K

The choice of K is critical for KNN performance. Let's explore how to find the optimal value of K using:

1. Cross-validation
2. Error plots
3. GridSearchCV

In [ ]:
# Find optimal K for classification
k_range = list(range(1, 31))
scores = []
cv_scores = []

# Train model with different k values and record performance
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_iris_scaled, y_train_iris)
    scores.append(knn.score(X_test_iris_scaled, y_test_iris))
    
    # Cross-validation score
    cv_score = cross_val_score(knn, X_train_iris_scaled, y_train_iris, cv=5).mean()
    cv_scores.append(cv_score)
    
# Plot the results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(k_range, scores, marker='o', markersize=5)
plt.title('Test Accuracy vs. K Value')
plt.xlabel('K Value')
plt.ylabel('Test Accuracy')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(k_range, cv_scores, marker='o', markersize=5, color='orange')
plt.title('Cross-Validation Accuracy vs. K Value')
plt.xlabel('K Value')
plt.ylabel('5-Fold CV Accuracy')
plt.grid(True)

plt.tight_layout()
plt.show()

# Find best K from cross-validation
best_k_cv = k_range[np.argmax(cv_scores)]
print(f"Best K value from cross-validation: {best_k_cv}")
print(f"Best accuracy score: {max(cv_scores):.4f}")

In [ ]:
# Use GridSearchCV to find optimal parameters for classification
param_grid = {
    'n_neighbors': list(range(1, 31)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True
)

grid_search.fit(X_train_iris_scaled, y_train_iris)

print("GridSearchCV Results for Classification:")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Get the results as a DataFrame
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results_filtered = cv_results[cv_results['param_metric'] == 'euclidean']

# Plot mean test scores for different n_neighbors and weights
plt.figure(figsize=(12, 6))
for weight in ['uniform', 'distance']:
    subset = cv_results_filtered[cv_results_filtered['param_weights'] == weight]
    plt.plot(subset['param_n_neighbors'], subset['mean_test_score'], 
             marker='o', label=f'weights={weight}')

plt.title('GridSearchCV: Performance vs. K Value and Weighting')
plt.xlabel('K Value')
plt.ylabel('5-Fold CV Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Find optimal K for regression
k_range = list(range(1, 31))
mse_scores = []
r2_scores = []
cv_mse = []

# Train model with different k values and record performance
for k in k_range:
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_train_reg_scaled, y_train_reg)
    
    # Predict and evaluate
    y_pred = knn_reg.predict(X_test_reg_scaled)
    mse_scores.append(mean_squared_error(y_test_reg, y_pred))
    r2_scores.append(r2_score(y_test_reg, y_pred))
    
    # Cross-validation MSE (negated for consistency with scoring in sklearn)
    cv_error = -cross_val_score(knn_reg, X_train_reg_scaled, y_train_reg, 
                               cv=5, scoring='neg_mean_squared_error').mean()
    cv_mse.append(cv_error)
    
# Plot the results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(k_range, mse_scores, marker='o', markersize=5)
plt.title('Test MSE vs. K Value')
plt.xlabel('K Value')
plt.ylabel('Mean Squared Error')
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(k_range, r2_scores, marker='o', markersize=5, color='green')
plt.title('Test R^2 vs. K Value')
plt.xlabel('K Value')
plt.ylabel('R^2 Score')
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(k_range, cv_mse, marker='o', markersize=5, color='orange')
plt.title('CV MSE vs. K Value')
plt.xlabel('K Value')
plt.ylabel('5-Fold CV MSE')
plt.grid(True)

plt.tight_layout()
plt.show()

# Find best K for regression
best_k_reg = k_range[np.argmin(cv_mse)]
print(f"Best K value for regression from cross-validation: {best_k_reg}")
print(f"Best MSE score: {min(cv_mse):.4f}")

## 7. Visualizing Decision Boundaries

Let's visualize how KNN creates decision boundaries for different K values in 2D feature space.

In [ ]:
# Create a function to plot decision boundaries
def plot_decision_boundary(X, y, classifier, title, ax=None):
    if ax is None:
        ax = plt.gca()
        
    # Define the mesh grid
    h = .02  # step size in the mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Get predictions for the entire mesh grid
    Z = classifier.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot the decision boundary
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])
    
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.8)
    
    # Plot the training points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold,
               edgecolor='k', s=40)
    
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    
    return ax


# Select only the first two features for visualization
X_iris_2d = X_iris[:, :2]  # sepal length and width
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_iris_2d, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# Scale the features
scaler_2d = StandardScaler()
X_train_2d_scaled = scaler_2d.fit_transform(X_train_2d)
X_test_2d_scaled = scaler_2d.transform(X_test_2d)

# Create figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Generate models with different K values
k_values = [1, 3, 5, 11, 21, 51]

for i, k in enumerate(k_values):
    # Create and train the classifier
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train_2d_scaled, y_train_2d)
    
    # Plot decision boundary
    plot_decision_boundary(
        X_train_2d_scaled, y_train_2d, clf, 
        f"K = {k}, Accuracy = {clf.score(X_test_2d_scaled, y_test_2d):.2f}",
        ax=axes[i]
    )

plt.tight_layout()
plt.show()

## 8. KNN for Regression Tasks

Now let's explore KNN for regression in more detail, comparing different approaches for averaging neighbor values.

In [ ]:
# Generate a more complex regression dataset
X_complex, y_complex = make_regression(n_samples=200, n_features=1, noise=25, 
                                       random_state=42, bias=50)
y_complex = y_complex + 10 * np.sin(X_complex.flatten() * 0.3)  # Add non-linearity

X_train_complex, X_test_complex, y_train_complex, y_test_complex = train_test_split(
    X_complex, y_complex, test_size=0.3, random_state=42
)

# Scale the features
scaler_complex = StandardScaler()
X_train_complex_scaled = scaler_complex.fit_transform(X_train_complex)
X_test_complex_scaled = scaler_complex.transform(X_test_complex)

# Create and train KNN regressor models with different weights and K values
k_values = [1, 3, 5, 9, 15]
weight_options = ['uniform', 'distance']

# Set up the plot
plt.figure(figsize=(15, 10))

for i, weights in enumerate(weight_options):
    for j, k in enumerate(k_values):
        # Create subplot
        plt.subplot(len(weight_options), len(k_values), 
                  i * len(k_values) + j + 1)
        
        # Create and train model
        knn_reg = KNeighborsRegressor(n_neighbors=k, weights=weights)
        knn_reg.fit(X_train_complex_scaled, y_train_complex)
        
        # Make predictions for a smooth curve
        X_plot = np.linspace(X_complex.min(), X_complex.max(), 1000).reshape(-1, 1)
        X_plot_scaled = scaler_complex.transform(X_plot)
        y_plot = knn_reg.predict(X_plot_scaled)
        
        # Evaluate the model
        y_pred = knn_reg.predict(X_test_complex_scaled)
        mse = mean_squared_error(y_test_complex, y_pred)
        r2 = r2_score(y_test_complex, y_pred)
        
        # Plot
        plt.scatter(X_train_complex, y_train_complex, s=10, alpha=0.4, label='Training data')
        plt.plot(X_plot, y_plot, color='red', linewidth=2, label='Prediction')
        plt.title(f"K={k}, Weights={weights}\nMSE={mse:.2f}, R²={r2:.2f}")
        plt.xlabel('X')
        plt.ylabel('y')
        plt.grid(True, alpha=0.3)
        
plt.tight_layout()
plt.show()

# Compare overall performance across different models
results = []

for weights in weight_options:
    for k in k_values:
        knn_reg = KNeighborsRegressor(n_neighbors=k, weights=weights)
        knn_reg.fit(X_train_complex_scaled, y_train_complex)
        y_pred = knn_reg.predict(X_test_complex_scaled)
        mse = mean_squared_error(y_test_complex, y_pred)
        r2 = r2_score(y_test_complex, y_pred)
        
        results.append({
            'k': k,
            'weights': weights,
            'MSE': mse,
            'R²': r2
        })

results_df = pd.DataFrame(results)
print("KNN Regression Performance Comparison:")
print(results_df)

# Plot MSE comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for weights in weight_options:
    subset = results_df[results_df['weights'] == weights]
    plt.plot(subset['k'], subset['MSE'], marker='o', label=f'Weights: {weights}')
plt.title('MSE vs K for Different Weight Types')
plt.xlabel('K Value')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for weights in weight_options:
    subset = results_df[results_df['weights'] == weights]
    plt.plot(subset['k'], subset['R²'], marker='o', label=f'Weights: {weights}')
plt.title('R² vs K for Different Weight Types')
plt.xlabel('K Value')
plt.ylabel('R² Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 9. Advantages and Limitations of KNN

### Advantages of KNN

1. **Simple and intuitive**: Easy to understand and implement
2. **No training phase**: The model simply stores the training data
3. **Non-parametric**: Makes no assumptions about the underlying data distribution
4. **Versatile**: Works for both classification and regression
5. **Naturally handles multi-class problems**: No need for special adaptations
6. **Effective for many practical applications**: Especially when the decision boundary is irregular

### Limitations of KNN

1. **Computationally expensive**: Need to calculate distances to all training samples for each prediction
2. **Memory intensive**: Requires storing all training data
3. **Curse of dimensionality**: Performance degrades with high-dimensional data
4. **Sensitive to irrelevant features**: All features contribute equally to distance calculation
5. **Sensitive to the scale of data**: Features need to be normalized
6. **Sensitive to outliers**: Especially with small values of k
7. **Optimal k selection**: Requires tuning through cross-validation

### Time Complexity

- Training: O(1) - Just storing the data
- Prediction: O(n × d) - For each test point, calculating distances to all n training points in d dimensions
- Space Complexity: O(n × d) - Storing all training data

### When to Use KNN

**Good use cases**:
- Small to medium-sized datasets
- Low-dimensional data
- When data has clear patterns in feature space
- When fast training is required

**Poor use cases**:
- Very large datasets
- High-dimensional data
- Datasets with many irrelevant features
- When fast prediction is critical

### Comparison with Other Algorithms

| Algorithm | Training Speed | Prediction Speed | Memory Usage | Interpretability | Handling of Non-linear Data |
|-----------|---------------|-----------------|-------------|-----------------|----------------------------|
| KNN       | Very Fast     | Slow            | High        | High            | Very Good                  |
| Decision Trees | Fast     | Very Fast       | Low         | High            | Good                       |
| Linear Regression | Fast  | Very Fast       | Low         | High            | Poor                       |
| SVM       | Moderate     | Fast            | Moderate    | Low             | Good (with kernels)        |
| Neural Networks | Slow    | Fast            | Moderate    | Very Low        | Very Good                  |

In [ ]:
# Demonstrating the impact of dimensionality on KNN performance
from sklearn.datasets import make_classification
from time import time

# Function to create datasets with varying dimensions and measure performance
def evaluate_knn_with_dimensions(dimensions, n_samples=1000, k=5):
    results = []
    
    for dim in dimensions:
        # Create dataset with specified dimensions
        X, y = make_classification(n_samples=n_samples, n_features=dim, 
                                  n_informative=int(dim*0.8), random_state=42)
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42
        )
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train KNN
        knn = KNeighborsClassifier(n_neighbors=k)
        
        # Measure training time
        start_time = time()
        knn.fit(X_train_scaled, y_train)
        train_time = time() - start_time
        
        # Measure prediction time
        start_time = time()
        y_pred = knn.predict(X_test_scaled)
        pred_time = time() - start_time
        
        # Calculate accuracy
        accuracy = accuracy_score(y_test, y_pred)
        
        results.append({
            'dimensions': dim,
            'accuracy': accuracy,
            'training_time': train_time,
            'prediction_time': pred_time
        })
    
    return pd.DataFrame(results)

# Test with different dimensions
dimensions = [2, 5, 10, 20, 50, 100, 200, 500]
knn_dim_results = evaluate_knn_with_dimensions(dimensions)

# Plot results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(knn_dim_results['dimensions'], knn_dim_results['accuracy'], marker='o')
plt.title('KNN Accuracy vs. Dimensionality')
plt.xlabel('Number of Features')
plt.ylabel('Accuracy')
plt.xscale('log')
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(knn_dim_results['dimensions'], knn_dim_results['training_time'], marker='o', color='green')
plt.title('KNN Training Time vs. Dimensionality')
plt.xlabel('Number of Features')
plt.ylabel('Training Time (s)')
plt.xscale('log')
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(knn_dim_results['dimensions'], knn_dim_results['prediction_time'], marker='o', color='red')
plt.title('KNN Prediction Time vs. Dimensionality')
plt.xlabel('Number of Features')
plt.ylabel('Prediction Time (s)')
plt.xscale('log')
plt.grid(True)

plt.tight_layout()
plt.show()

print("KNN Performance with Different Dimensionalities:")
print(knn_dim_results)

print("\n--- Key Findings ---")
print("1. As dimensionality increases, KNN tends to lose accuracy (curse of dimensionality)")
print("2. Training time increases slightly with dimensions (due to storing more data)")
print("3. Prediction time increases significantly with dimensionality (distance calculations)")

## Conclusion

In this notebook, we've explored the K-Nearest Neighbors algorithm in depth, covering both theoretical concepts and practical implementations. We've learned that:

1. **KNN is intuitive**: It classifies points by their neighbors' majority vote or predicts values by averaging nearby points.

2. **Parameter tuning is crucial**: The choice of K greatly affects model performance, with small K values leading to high variance and large K values causing high bias.

3. **Distance metrics matter**: Different distance metrics (Euclidean, Manhattan) can lead to different results depending on the data.

4. **Preprocessing is essential**: Feature scaling is critical for KNN since it relies on distance calculations.

5. **Dimensionality affects performance**: KNN suffers from the curse of dimensionality, as shown in our experiments.

6. **Use cases**: KNN works best with low-dimensional data where patterns are clear in feature space.

Despite its limitations with large datasets and high-dimensional data, KNN remains a valuable algorithm in a data scientist's toolkit, especially for benchmark comparisons and when interpretability is important.

### Further Reading
- "Pattern Recognition and Machine Learning" by Christopher M. Bishop
- "The Elements of Statistical Learning" by Hastie, Tibshirani, and Friedman
- Scikit-learn documentation: https://scikit-learn.org/stable/modules/neighbors.html